In [4]:
import pandas as pd
import numpy as np

DATASETS = ["cars3d", "dsprites", "mpi3d", "clevr", "iraven", "shapes3d"]

# Diccionarios para despliegue (ajústalos a tu gusto)
MODEL_NAME_MAP = {
    "resnet18": "R18",
    "resnet18_mixer": "LICG(R18)",
    "split": "AIN",
    "split_4x": "AIN (4x)",
    "split_resnet_mixer": "LCIG(AIN)",
    "split_resnet_mixer_red_64": "LCIG(AIN)-64",
    "split_resnet_mixer_red_128": "LCIG(AIN)-128",
    "split_resnet_mixer_red_256": "LCIG(AIN)-256",
    "split_resnet_mixer_iid": "LCIG(AIN)+IID",
    "split_resnet_mixer_no_mixer": "LCIG(AIN) - MIXER",
    "split_resnet_mixer_no_mixer_iid": "LCIG(AIN) - MIXER + IID",
    "split_resnet_mixer_all_cases": "LCIG(AIN) [ALL]",
    "split_resnet_mixer_all_cases_iid": "LCIG(AIN) [ALL] + IID",
    "ed": "ED",
}

DATASET_NAME_MAP = {
    "cars3d": "C3D",
    "dsprites": "dSprites",
    "mpi3d": "MPI3D",
    "clevr": "CLEVR",
    "iraven": "I-RAVEN",
    "shapes3d": "Sh3D",
}

# Columnas de métricas (para excluirlas al definir "config")
METRIC_COLS = ["train_acc", "val_acc", "ood_val_0_acc", "test_acc"]

DECIMALS = 2
BOLD_TOL = 1e-12  # tolerancia para empates/floating
def fmt(mean, std, decimals=DECIMALS):
    if pd.isna(mean):
        return ""
    if pd.isna(std):
        return f"{mean:.{decimals}f} (—)"
    return f"{mean:.{decimals}f} ({std:.{decimals}f})"


In [5]:
records = []

for dataset in DATASETS:
    df = pd.read_pickle(f"{dataset}_id.pkl").copy()

    # Nos aseguramos que existan
    if "arch" not in df.columns or "seed" not in df.columns or "test_acc" not in df.columns:
        raise ValueError(f"{dataset}_oracle.pkl debe contener columnas: arch, seed, test_acc")

    # Definimos qué significa "configuración": todas las columnas que NO son métricas,
    # excluyendo seed (porque seed la promediamos) pero incluyendo arch.
    non_metric_cols = [c for c in df.columns if c not in METRIC_COLS]
    config_cols = [c for c in non_metric_cols if c != "seed"]  # incluye "arch" y otros (si existen)

    # 1) Promedio por seed dentro de cada configuración
    per_seed = (
        df.groupby(config_cols + ["seed"], dropna=False)["test_acc"]
        .mean()
        .reset_index()
    )

    # 2) Estadísticos entre seeds por configuración
    stats = (
        per_seed.groupby(config_cols, dropna=False)["test_acc"]
        .agg(mean="mean", std="std", n="count")
        .reset_index()
    )

    # 3) Para cada arch, tomamos la configuración con mayor mean(test_acc)
    best_idx = stats.groupby("arch")["mean"].idxmax()
    best = stats.loc[best_idx].copy()

    # Guardamos largo: (arch, dataset) -> mean/std
    for _, r in best.iterrows():
        records.append(
            {"arch": r["arch"], "dataset": dataset, "mean": r["mean"], "std": r["std"]}
        )

# ---- DataFrame largo ----
long_df = pd.DataFrame(records)

# ---- matrices numéricas mean/std ----
mean_df = long_df.pivot(index="arch", columns="dataset", values="mean")
std_df = long_df.pivot(index="arch", columns="dataset", values="std")

# Mantener orden de datasets
mean_df = mean_df.reindex(columns=DATASETS)
std_df = std_df.reindex(columns=DATASETS)

'''
# =========================
# AGREGAR FILA (nuevo arch)
# =========================
new_arch = "ED"

# valores por dataset (usa keys = nombres originales en DATASETS)
new_mean = {
    "cars3d": 46.70,
    "dsprites": 61.31,
    "mpi3d": 70.76,
    "clevr": 53.98,
    "iraven": 76.12,
    "shapes3d": 96.09,
}
new_std = {
    "cars3d": 0.98,
    "dsprites": 0.61,
    "mpi3d": 0.24,
    "clevr": 1.46,
    "iraven": 2.70,
    "shapes3d": 2.65,
}

# añade la fila (alineando por columnas)
mean_df.loc[new_arch] = pd.Series(new_mean).reindex(mean_df.columns)
std_df.loc[new_arch]  = pd.Series(new_std).reindex(std_df.columns)

# (opcional) si quieres que se muestre con nombre bonito
MODEL_NAME_MAP[new_arch] = "ED"
'''

# ---- máscara de máximos por dataset ----
max_by_dataset = mean_df.max(axis=0, skipna=True)
is_max = (mean_df.sub(max_by_dataset, axis=1).abs() <= BOLD_TOL)

# ---- tabla base (strings sin markup) ----
plain = pd.DataFrame(index=mean_df.index, columns=mean_df.columns, dtype=object)
for ds in mean_df.columns:
    for arch in mean_df.index:
        plain.loc[arch, ds] = fmt(
            mean_df.loc[arch, ds], std_df.loc[arch, ds], decimals=DECIMALS
        )

# =========================
# NOTEBOOK DISPLAY (Styler)
# =========================
plain_disp = plain.rename(index=MODEL_NAME_MAP, columns=DATASET_NAME_MAP)

is_max_disp = (
    is_max.rename(index=MODEL_NAME_MAP, columns=DATASET_NAME_MAP)
    .reindex(index=plain_disp.index, columns=plain_disp.columns)
    .fillna(False)
)

# display() existe en notebooks; si estás en script puro, comenta estas líneas
display(
    plain_disp.style.apply(
        lambda col: [
            "font-weight: bold" if bool(is_max_disp.loc[idx, col.name]) else ""
            for idx in col.index
        ],
        axis=0,
    )
)

# =========================
# MARKDOWN
# =========================
md = plain.copy()
for ds in md.columns:
    for arch in md.index:
        if bool(is_max.loc[arch, ds]) and md.loc[arch, ds] != "":
            md.loc[arch, ds] = f"**{md.loc[arch, ds]}**"

md = md.rename(index=MODEL_NAME_MAP, columns=DATASET_NAME_MAP)
print(md.to_markdown())

# =========================
# LATEX (con achique)
# =========================
tex = plain.copy()
for ds in tex.columns:
    for arch in tex.index:
        if bool(is_max.loc[arch, ds]) and tex.loc[arch, ds] != "":
            tex.loc[arch, ds] = r"\textbf{" + tex.loc[arch, ds] + "}"

tex = tex.rename(index=MODEL_NAME_MAP, columns=DATASET_NAME_MAP)

# (opcional) limpiar nombres para evitar header doble "dataset"/"arch"
tex.index.name = None
tex.columns.name = None

latex_tabular = tex.to_latex(escape=False, index=True)

latex_small = (
    r"\begin{table}[t]" "\n"
    r"\centering" "\n"
    r"\small" "\n"
    r"\setlength{\tabcolsep}{4pt}" "\n"
    r"\renewcommand{\arraystretch}{1.1}" "\n"
    r"\resizebox{\textwidth}{!}{%" "\n"
    + latex_tabular + "\n"
    r"}" "\n"
    r"\caption{TODO: caption}" "\n"
    r"\label{tab:oracle_results}" "\n"
    r"\end{table}"
)

print(latex_small)

dataset,C3D,dSprites,MPI3D,CLEVR,I-RAVEN,Sh3D
arch,,,,,,
ED,45.44 (1.56),63.12 (1.79),59.51 (5.82),53.39 (3.04),74.89 (3.64),98.15 (1.20)
R18,32.12 (1.55),20.76 (0.93),42.82 (0.88),27.14 (6.72),11.26 (3.49),86.44 (1.05)
LICG(R18),41.97 (2.42),22.15 (2.68),44.95 (2.15),13.34 (—),9.60 (8.84),91.39 (2.08)
AIN,43.56 (0.96),61.98 (1.80),55.73 (0.31),53.77 (2.60),63.63 (1.96),85.02 (1.61)
split_1,,,60.95 (1.57),,,
split_2,,,69.19 (0.75),,,
split_3,,,72.01 (1.38),,,
split_4,,,61.56 (6.19),,,
AIN (4x),42.62 (1.82),60.02 (2.17),47.81 (1.85),50.69 (3.06),59.63 (2.94),80.54 (1.55)


| arch                                  | C3D              | dSprites         | MPI3D            | CLEVR            | I-RAVEN          | Sh3D             |
|:--------------------------------------|:-----------------|:-----------------|:-----------------|:-----------------|:-----------------|:-----------------|
| ED                                    | 45.44 (1.56)     | 63.12 (1.79)     | 59.51 (5.82)     | 53.39 (3.04)     | **74.89 (3.64)** | **98.15 (1.20)** |
| R18                                   | 32.12 (1.55)     | 20.76 (0.93)     | 42.82 (0.88)     | 27.14 (6.72)     | 11.26 (3.49)     | 86.44 (1.05)     |
| LICG(R18)                             | 41.97 (2.42)     | 22.15 (2.68)     | 44.95 (2.15)     | 13.34 (—)        | 9.60 (8.84)      | 91.39 (2.08)     |
| AIN                                   | 43.56 (0.96)     | 61.98 (1.80)     | 55.73 (0.31)     | 53.77 (2.60)     | 63.63 (1.96)     | 85.02 (1.61)     |
| split_1                               |                  |    

In [6]:
records = []

for dataset in DATASETS:
    df = pd.read_pickle(f"{dataset}_oracle.pkl").copy()

    # Nos aseguramos que existan
    if "arch" not in df.columns or "seed" not in df.columns or "test_acc" not in df.columns:
        raise ValueError(f"{dataset}_oracle.pkl debe contener columnas: arch, seed, test_acc")

    # Definimos qué significa "configuración": todas las columnas que NO son métricas,
    # excluyendo seed (porque seed la promediamos) pero incluyendo arch.
    non_metric_cols = [c for c in df.columns if c not in METRIC_COLS]
    config_cols = [c for c in non_metric_cols if c != "seed"]  # incluye "arch" y otros (si existen)

    # 1) Promedio por seed dentro de cada configuración
    per_seed = (
        df.groupby(config_cols + ["seed"], dropna=False)["test_acc"]
        .mean()
        .reset_index()
    )

    # 2) Estadísticos entre seeds por configuración
    stats = (
        per_seed.groupby(config_cols, dropna=False)["test_acc"]
        .agg(mean="mean", std="std", n="count")
        .reset_index()
    )

    # 3) Para cada arch, tomamos la configuración con mayor mean(test_acc)
    best_idx = stats.groupby("arch")["mean"].idxmax()
    best = stats.loc[best_idx].copy()

    # Guardamos largo: (arch, dataset) -> mean/std
    for _, r in best.iterrows():
        records.append(
            {"arch": r["arch"], "dataset": dataset, "mean": r["mean"], "std": r["std"]}
        )

# ---- DataFrame largo ----
long_df = pd.DataFrame(records)

# ---- matrices numéricas mean/std ----
mean_df = long_df.pivot(index="arch", columns="dataset", values="mean")
std_df = long_df.pivot(index="arch", columns="dataset", values="std")

# Mantener orden de datasets
mean_df = mean_df.reindex(columns=DATASETS)
std_df = std_df.reindex(columns=DATASETS)

# ---- máscara de máximos por dataset ----
max_by_dataset = mean_df.max(axis=0, skipna=True)
is_max = (mean_df.sub(max_by_dataset, axis=1).abs() <= BOLD_TOL)

# ---- tabla base (strings sin markup) ----
plain = pd.DataFrame(index=mean_df.index, columns=mean_df.columns, dtype=object)
for ds in mean_df.columns:
    for arch in mean_df.index:
        plain.loc[arch, ds] = fmt(
            mean_df.loc[arch, ds], std_df.loc[arch, ds], decimals=DECIMALS
        )

# =========================
# NOTEBOOK DISPLAY (Styler)
# =========================
plain_disp = plain.rename(index=MODEL_NAME_MAP, columns=DATASET_NAME_MAP)

is_max_disp = (
    is_max.rename(index=MODEL_NAME_MAP, columns=DATASET_NAME_MAP)
    .reindex(index=plain_disp.index, columns=plain_disp.columns)
    .fillna(False)
)

# display() existe en notebooks; si estás en script puro, comenta estas líneas
display(
    plain_disp.style.apply(
        lambda col: [
            "font-weight: bold" if bool(is_max_disp.loc[idx, col.name]) else ""
            for idx in col.index
        ],
        axis=0,
    )
)

# =========================
# MARKDOWN
# =========================
md = plain.copy()
for ds in md.columns:
    for arch in md.index:
        if bool(is_max.loc[arch, ds]) and md.loc[arch, ds] != "":
            md.loc[arch, ds] = f"**{md.loc[arch, ds]}**"

md = md.rename(index=MODEL_NAME_MAP, columns=DATASET_NAME_MAP)
print(md.to_markdown())

# =========================
# LATEX (con achique)
# =========================
tex = plain.copy()
for ds in tex.columns:
    for arch in tex.index:
        if bool(is_max.loc[arch, ds]) and tex.loc[arch, ds] != "":
            tex.loc[arch, ds] = r"\textbf{" + tex.loc[arch, ds] + "}"

tex = tex.rename(index=MODEL_NAME_MAP, columns=DATASET_NAME_MAP)

# (opcional) limpiar nombres para evitar header doble "dataset"/"arch"
tex.index.name = None
tex.columns.name = None

latex_tabular = tex.to_latex(escape=False, index=True)

latex_small = (
    r"\begin{table}[t]" "\n"
    r"\centering" "\n"
    r"\small" "\n"
    r"\setlength{\tabcolsep}{4pt}" "\n"
    r"\renewcommand{\arraystretch}{1.1}" "\n"
    r"\resizebox{\textwidth}{!}{%" "\n"
    + latex_tabular + "\n"
    r"}" "\n"
    r"\caption{TODO: caption}" "\n"
    r"\label{tab:oracle_results}" "\n"
    r"\end{table}"
)

print(latex_small)

dataset,C3D,dSprites,MPI3D,CLEVR,I-RAVEN,Sh3D
arch,,,,,,
ED,50.36 (1.29),64.29 (1.32),62.37 (7.98),64.49 (4.05),84.23 (3.20),98.85 (0.52)
R18,35.85 (1.35),22.71 (0.76),43.59 (0.22),30.70 (4.54),17.98 (7.03),86.44 (1.05)
LICG(R18),44.68 (1.50),23.97 (2.50),45.95 (3.49),39.33 (—),17.82 (4.93),92.11 (1.66)
AIN,46.27 (1.33),62.06 (1.66),55.80 (0.34),61.76 (2.38),76.96 (3.25),87.56 (2.28)
split_1,,,61.40 (1.07),,,
split_2,,,70.84 (0.32),,,
split_3,,,73.72 (0.71),,,
split_4,,,65.01 (5.77),,,
AIN (4x),45.15 (0.64),61.23 (2.44),48.02 (1.95),59.44 (3.58),69.55 (3.77),89.06 (0.20)


| arch                                  | C3D              | dSprites         | MPI3D            | CLEVR            | I-RAVEN          | Sh3D             |
|:--------------------------------------|:-----------------|:-----------------|:-----------------|:-----------------|:-----------------|:-----------------|
| ED                                    | 50.36 (1.29)     | 64.29 (1.32)     | 62.37 (7.98)     | 64.49 (4.05)     | 84.23 (3.20)     | **98.85 (0.52)** |
| R18                                   | 35.85 (1.35)     | 22.71 (0.76)     | 43.59 (0.22)     | 30.70 (4.54)     | 17.98 (7.03)     | 86.44 (1.05)     |
| LICG(R18)                             | 44.68 (1.50)     | 23.97 (2.50)     | 45.95 (3.49)     | 39.33 (—)        | 17.82 (4.93)     | 92.11 (1.66)     |
| AIN                                   | 46.27 (1.33)     | 62.06 (1.66)     | 55.80 (0.34)     | 61.76 (2.38)     | 76.96 (3.25)     | 87.56 (2.28)     |
| split_1                               |                  |    